

# Data-Driven Atmospheric Radiative Transfer Inversion Using AI

**Author:** Cristina Sgattoni  
**Original framework:** Matthias Chung (2024)

This notebook presents a neural network-based approach to atmospheric radiative transfer inversion using the **Latent Twins architecture**.

Two autoencoders and two linear latent-space mappings are jointly trained to learn forward and inverse relationships between atmospheric states and radiative observations.

After training, the main models are frozen, and two auxiliary neural networks are trained to correct inconsistencies in retrieved liquid and ice cloud microphysical properties.

The notebook illustrates the complete two-stage training procedure using preprocessed and normalized atmospheric data.

### Data Interface

The notebook requires four preprocessed and normalized arrays: `X_train`, `Y_train`, `X_test`, and `Y_test`, with samples arranged in rows.

- **Atmospheric states (`X`):** Skin temperature, 60-level profiles of temperature, water vapour and ozone, 301 surface emissivity values, and four 60-level cloud variables (LWC, REL, IWC, REI), in this order.

- **Radiative observations (`Y`):** Longitude, latitude, month, hour, 60 pressure levels, and radiance spectra, in this order.

All quantities provided to the networks are **normalized and dimensionless**. Preprocessing and normalization are not included in this notebook.


In [ ]:
from pathlib import Path
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import scipy.io

MODEL_DIR = Path("model")
RESULTS_DIR = Path("results")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 512
MAIN_EPOCHS = 10000
N_LATENT_X = 512
N_LATENT_Y = 512
LEARNING_RATE = 1e-3
LR_STEP_SIZE = 500
LR_GAMMA = 0.75
WEIGHT_XX = 1.0
WEIGHT_YY = 1.0
WEIGHT_XY = 0.05
WEIGHT_YX = 1.0
PRINT_EVERY = 1
LOAD_EXISTING_MODELS = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
N_LEVELS = 60
N_EMISSIVITY = 301
AUX_EPOCHS = 1000
AUX_LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
LR_FACTOR = 0.5
LR_PATIENCE = 5
INCOHERENT_WEIGHT = 5.0
COHERENT_WEIGHT = 1.0
MISMATCH_WEIGHT = 1.0
MEAN_SHIFT_WEIGHT = 5.0
EPS_LWC = EPS_REL = EPS_IWC = EPS_REI = 0.0
PRINT_BATCH_SHAPES = False
AUX_DEVICE = torch.device("cpu")  # Auxiliary training and evaluation run on CPU

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


## 1. Input Data

The notebook requires four preprocessed and normalized arrays: `X_train`, `Y_train`, `X_test`, and `Y_test`.

The arrays can be provided as NumPy arrays or PyTorch tensors and must have shape `(n_samples, n_features)`, with samples arranged in rows and variables in columns.

Data loading and preprocessing are not included in this notebook.

In [ ]:
# Supply these four arrays in a preceding cell or notebook environment:
# X_train, Y_train, X_test, Y_test

for name in ("X_train", "Y_train", "X_test", "Y_test"):
    if name not in globals():
        raise NameError(f"Provide the normalized array {name} before running this cell.")

class PairedDataset(Dataset):
    def __init__(self, x, y):
        inputs = torch.as_tensor(np.asarray(x), dtype=torch.float32)
        targets = torch.as_tensor(np.asarray(y), dtype=torch.float32)
        if inputs.ndim != 2 or targets.ndim != 2 or inputs.shape[0] != targets.shape[0]:
            raise ValueError("X and Y must be two-dimensional arrays with matching case counts.")
        self.inputs, self.targets = inputs, targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

train_dataset = PairedDataset(X_train, Y_train)
test_dataset = PairedDataset(X_test, Y_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
input_dim_x = train_dataset.inputs.shape[1]
input_dim_y = train_dataset.targets.shape[1]
if input_dim_x != 1 + 7 * N_LEVELS + N_EMISSIVITY:
    raise ValueError("X dimensions are inconsistent with the required atmospheric variable layout.")
if input_dim_y < 4 + N_LEVELS:
    raise ValueError("Y does not contain the expected pressure profile.")
print("Training cases:", len(train_dataset), "Test cases:", len(test_dataset))
print("Dimensions:", input_dim_x, input_dim_y)

## 2. Main model architectures

In [ ]:
class FlexibleAutoencoder(nn.Module):
    """
    Fully connected autoencoder with configurable hidden dimensions.

    The encoder maps the original input into a low-dimensional
    latent representation.

    The decoder reconstructs the original input from the latent
    representation.

    ReLU activations are applied after each linear layer,
    including the final decoder layer, as in the original model.

    Parameters
    ----------
    input_dim : int
        Dimension of the input vector.

    hidden_dims : list of int
        Dimensions of the encoder hidden layers.
        The last element defines the latent dimension.

    output_dim : int
        Dimension of the reconstructed output vector.
    """

    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()

        # Encoder
        encoder_layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            encoder_layers.append(
                nn.Linear(prev_dim, hidden_dim)
            )
            encoder_layers.append(nn.ReLU(inplace=True))
            prev_dim = hidden_dim

        self.encoder = nn.Sequential(*encoder_layers)

        # Decoder
        decoder_layers = []

        for hidden_dim in reversed(hidden_dims):
            decoder_layers.append(
                nn.Linear(prev_dim, hidden_dim)
            )
            decoder_layers.append(nn.ReLU(inplace=True))
            prev_dim = hidden_dim

        decoder_layers.append(
            nn.Linear(prev_dim, output_dim)
        )
        decoder_layers.append(nn.ReLU(inplace=True))

        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)

        return reconstructed, latent

class LinearLatentSpaceMapping(nn.Module):
    """
    Linear mapping between the latent spaces of two autoencoders.

    The mapping is learned jointly with the autoencoders.

    Parameters
    ----------
    input_dim : int
        Dimension of the source latent space.

    output_dim : int
        Dimension of the target latent space.
    """

    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)


## 3. Initialization and Joint Training of the Main Models

The two autoencoders and the two latent-space mappings are jointly trained by minimizing a weighted combination of four Mean Squared Error (MSE) terms: atmospheric state reconstruction, observation reconstruction, forward prediction, and inverse prediction.

The training loss history records the loss values from the final minibatch of each epoch.


In [ ]:
# ---------------------------------------------------------------
# INITIALIZE AUTOENCODERS
# ---------------------------------------------------------------

hidden_dims_x = [
    int((input_dim_x + N_LATENT_X) / 2),
    N_LATENT_X
]

hidden_dims_y = [
    int((input_dim_y + N_LATENT_Y) / 2),
    N_LATENT_Y
]

model_x = FlexibleAutoencoder(
    input_dim_x,
    hidden_dims_x,
    input_dim_x
)

model_y = FlexibleAutoencoder(
    input_dim_y,
    hidden_dims_y,
    input_dim_y
)

# Linear latent mappings
mapping_xy = LinearLatentSpaceMapping(
    N_LATENT_X,
    N_LATENT_Y
)

mapping_yx = LinearLatentSpaceMapping(
    N_LATENT_Y,
    N_LATENT_X
)

# ---------------------------------------------------------------
# LOAD EXISTING MODEL PARAMETERS
# ---------------------------------------------------------------

model_files = {
    "model_x": MODEL_DIR / "modelX.pth",
    "model_y": MODEL_DIR / "modelY.pth",
    "mapping_xy": MODEL_DIR / "mapping_modelXY.pth",
    "mapping_yx": MODEL_DIR / "mapping_modelYX.pth"
}

history_file = MODEL_DIR / "trainingHistory.pth"

checkpoints_exist = (
    all(path.exists() for path in model_files.values())
)

load_main_checkpoints = LOAD_EXISTING_MODELS and checkpoints_exist

if load_main_checkpoints:

    print("Loading existing model parameters...")

    model_x.load_state_dict(
        torch.load(
            model_files["model_x"],
            map_location=DEVICE,
            weights_only=True
        )
    )

    model_y.load_state_dict(
        torch.load(
            model_files["model_y"],
            map_location=DEVICE,
            weights_only=True
        )
    )

    mapping_xy.load_state_dict(
        torch.load(
            model_files["mapping_xy"],
            map_location=DEVICE,
            weights_only=True
        )
    )

    mapping_yx.load_state_dict(
        torch.load(
            model_files["mapping_yx"],
            map_location=DEVICE,
            weights_only=True
        )
    )

else:
    print("Initializing new models.")

# Move models to the selected device
model_x = model_x.to(DEVICE)
model_y = model_y.to(DEVICE)

mapping_xy = mapping_xy.to(DEVICE)
mapping_yx = mapping_yx.to(DEVICE)

# ---------------------------------------------------------------
# DEFINE LOSS FUNCTIONS
# ---------------------------------------------------------------

criterion_xx = nn.MSELoss()
criterion_yy = nn.MSELoss()
criterion_xy = nn.MSELoss()
criterion_yx = nn.MSELoss()

# ---------------------------------------------------------------
# DEFINE JOINT OPTIMIZER
# ---------------------------------------------------------------

# All four models are trained simultaneously.
theta = (
    list(model_x.parameters())
    + list(model_y.parameters())
    + list(mapping_xy.parameters())
    + list(mapping_yx.parameters())
)

joint_optimizer = optim.Adam(
    theta,
    lr=LEARNING_RATE
)

scheduler = optim.lr_scheduler.StepLR(
    joint_optimizer,
    step_size=LR_STEP_SIZE,
    gamma=LR_GAMMA
)

# ---------------------------------------------------------------
# INITIALIZE TRAINING HISTORY
# ---------------------------------------------------------------

if load_main_checkpoints and history_file.exists():

    print("Loading training history...")

    training_history = torch.load(
        history_file,
        map_location="cpu",
        weights_only=True
    )

else:

    training_history = {
        "lossesXX": [],
        "lossesYY": [],
        "lossesXY": [],
        "lossesYX": [],
        "losses": []
    }

# ---------------------------------------------------------------
# JOINT TRAINING
# ---------------------------------------------------------------

print("-" * 60)
print("Starting training...")

model_x.train()
model_y.train()
mapping_xy.train()
mapping_yx.train()

for epoch in range(MAIN_EPOCHS):

    for x_batch, y_batch in train_loader:

        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        # -------------------------------------------------------
        # Forward pass: autoencoder reconstructions
        # -------------------------------------------------------

        auto_x, latent_x = model_x(x_batch)
        auto_y, latent_y = model_y(y_batch)

        # -------------------------------------------------------
        # Linear mappings between latent spaces
        # -------------------------------------------------------

        latent_x_to_y = mapping_xy(latent_x)
        latent_y_to_x = mapping_yx(latent_y)

        # -------------------------------------------------------
        # Surrogate forward and inverse predictions
        # -------------------------------------------------------

        predicted_xy = model_y.decoder(
            latent_x_to_y
        )

        predicted_yx = model_x.decoder(
            latent_y_to_x
        )

        # -------------------------------------------------------
        # Compute the four loss components
        # -------------------------------------------------------

        loss_xx = criterion_xx(
            auto_x, x_batch
        )

        loss_yy = criterion_yy(
            auto_y, y_batch
        )

        loss_xy = criterion_xy(
            predicted_xy, y_batch
        )

        loss_yx = criterion_yx(
            predicted_yx, x_batch
        )

        # -------------------------------------------------------
        # Weighted joint objective
        # -------------------------------------------------------

        total_loss = (
            WEIGHT_XX * loss_xx
            + WEIGHT_YY * loss_yy
            + WEIGHT_XY * loss_xy
            + WEIGHT_YX * loss_yx
        )

        # -------------------------------------------------------
        # Backpropagation and optimization
        # -------------------------------------------------------

        joint_optimizer.zero_grad()
        total_loss.backward()
        joint_optimizer.step()

    # Update learning rate after each epoch
    scheduler.step()

    # Record losses from the final batch of the epoch
    training_history["lossesXX"].append(
        loss_xx.item()
    )

    training_history["lossesYY"].append(
        loss_yy.item()
    )

    training_history["lossesXY"].append(
        loss_xy.item()
    )

    training_history["lossesYX"].append(
        loss_yx.item()
    )

    training_history["losses"].append(
        total_loss.item()
    )

    # Print training progress
    if (epoch + 1) % PRINT_EVERY == 0:

        current_lr = (
            joint_optimizer.param_groups[0]["lr"]
        )

        print(
            f"Epoch [{epoch + 1}/{MAIN_EPOCHS}] "
            f"Loss: {total_loss.item():.4e} | "
            f"XX: {loss_xx.item():.4e} | "
            f"YY: {loss_yy.item():.4e} | "
            f"XY: {loss_xy.item():.4e} | "
            f"YX: {loss_yx.item():.4e} | "
            f"LR: {current_lr:.2e}"
        )

print("Training completed.")

# ---------------------------------------------------------------
# SAVE TRAINING HISTORY
# ---------------------------------------------------------------

torch.save(
    training_history,
    history_file
)

# ---------------------------------------------------------------
# SAVE TRAINED MODELS
# ---------------------------------------------------------------

print("Saving trained models...")

torch.save(
    model_x.state_dict(),
    model_files["model_x"]
)

torch.save(
    model_y.state_dict(),
    model_files["model_y"]
)

torch.save(
    mapping_xy.state_dict(),
    model_files["mapping_xy"]
)

torch.save(
    mapping_yx.state_dict(),
    model_files["mapping_yx"]
)


## 4. Freezing the Pretrained Models

After joint training, the parameters of the two autoencoders and the two latent-space mappings are frozen to prevent further updates during the training of the auxiliary networks.

The pretrained models are moved to the CPU for the second training stage.


In [ ]:
for model in (model_x, model_y, mapping_xy, mapping_yx):
    model.to(AUX_DEVICE)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.eval()

DEVICE = AUX_DEVICE  # Used by the original evaluation helper functions below

## 5. Cloud Correction Architectures and Loss Function

In [ ]:
class SimpleNN(nn.Module):
    """
    Neural network for cloud microphysical correction.

    Input dimension
    ---------------
    5 normalized variables per atmospheric level.

    Output dimension
    ----------------
    2 corrected cloud variables.

    Architecture
    ------------
    5 -> 4 -> 3 -> 2

    Softplus activations are used after each linear layer,
    including the output layer.

    The architecture is identical for liquid and ice clouds.
    """

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(5, 4),
            nn.Softplus(),
            nn.Linear(4, 3),
            nn.Softplus(),
            nn.Linear(3, 2),
            nn.Softplus()
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def incoherence_mask(
    var1,
    var2,
    eps1=0.0,
    eps2=0.0
):
    """
    Identify inconsistent cloud-variable pairs.

    An inconsistency occurs when one variable is considered zero
    while the other is considered nonzero.

    Zero is defined using the absolute-value thresholds eps1
    and eps2.
    """

    zero1 = var1.abs() <= eps1
    zero2 = var2.abs() <= eps2

    return (zero1 & ~zero2) | (zero2 & ~zero1)

In [ ]:
def count_incoherences(var1, var2):
    """
    Count exact zero/nonzero inconsistencies.

    This reproduces the original diagnostic based on equality
    to zero, independently of the configurable test thresholds.
    """

    mask = ((var1 == 0) & (var2 != 0)) | (
        (var2 == 0) & (var1 != 0)
    )

    return mask.sum().item()

In [ ]:
def zero_nonzero_mismatch(var1, var2):
    """
    Compute the fraction of zero/nonzero mismatches.

    Note
    ----
    Exact equality comparisons are not differentiable.
    Consequently, this term does not provide gradients to
    the corrective networks.
    """

    mask1 = (var1 == 0).float()
    mask2 = (var2 == 0).float()

    return torch.mean(torch.abs(mask1 - mask2))

In [ ]:
def extract_cloud_variables(x, y):
    """
    Extract cloud and auxiliary atmospheric variables.

    The atmospheric state variables are ordered as follows:

        Tskin, T, WV, O3, emissivity, LWC, REL, IWC, REI.

    Returns
    -------
    dict
        Arrays of shape (batch_size, N_LEVELS).

    All returned quantities are normalized and dimensionless.
    """

    nlay = N_LEVELS
    nemi = N_EMISSIVITY

    cloud_start = 1 + 3 * nlay + nemi

    return {
        "lwc": x[:, cloud_start:cloud_start + nlay],
        "rel": x[:, cloud_start + nlay:cloud_start + 2 * nlay],
        "iwc": x[:, cloud_start + 2 * nlay:cloud_start + 3 * nlay],
        "rei": x[:, cloud_start + 3 * nlay:cloud_start + 4 * nlay],
        "tem": x[:, 1:1 + nlay],
        "wv": x[:, 1 + nlay:1 + 2 * nlay],
        "press": y[:, 4:4 + nlay]
    }

In [ ]:
def prepare_corrective_inputs(variables):
    """
    Prepare one five-dimensional input vector per atmospheric level.

    Liquid input:
        [LWC, REL, temperature, water vapour, pressure]

    Ice input:
        [IWC, REI, temperature, water vapour, pressure]

    Output shape
    ------------
    (batch_size * N_LEVELS, 5)
    """

    input_liquid = torch.stack(
        [
            variables["lwc"],
            variables["rel"],
            variables["tem"],
            variables["wv"],
            variables["press"]
        ],
        dim=2
    ).reshape(-1, 5)

    input_ice = torch.stack(
        [
            variables["iwc"],
            variables["rei"],
            variables["tem"],
            variables["wv"],
            variables["press"]
        ],
        dim=2
    ).reshape(-1, 5)

    return input_liquid, input_ice

In [ ]:
def compute_corrections(
    variables,
    network_liquid,
    network_ice
):
    """
    Compute corrected liquid and ice cloud variables.

    Both networks are evaluated at all atmospheric levels.
    Corrections are subsequently applied only at inconsistent
    points during evaluation.
    """

    input_liquid, input_ice = prepare_corrective_inputs(
        variables
    )

    output_liquid = network_liquid(input_liquid)
    output_ice = network_ice(input_ice)

    corrected = {
        "lwc": output_liquid[:, 0].reshape_as(variables["lwc"]),
        "rel": output_liquid[:, 1].reshape_as(variables["rel"]),
        "iwc": output_ice[:, 0].reshape_as(variables["iwc"]),
        "rei": output_ice[:, 1].reshape_as(variables["rei"])
    }

    return corrected

In [ ]:
def compute_corrective_loss(
    corrected,
    targets,
    mask_liquid,
    mask_ice,
    criterion
):
    """
    Compute the weighted corrective-network loss.

    Reconstruction errors are evaluated separately on
    inconsistent and consistent points.

    The mean-shift penalty discourages systematic differences
    between corrected and reference cloud variables.

    The Boolean mismatch term is non-differentiable.
    """

    loss = torch.tensor(
        0.0,
        device=corrected["lwc"].device
    )

    cloud_pairs = [
        ("lwc", "rel", mask_liquid),
        ("iwc", "rei", mask_ice)
    ]

    for var1, var2, mask in cloud_pairs:

        if mask.any():
            loss = loss + INCOHERENT_WEIGHT * criterion(
                corrected[var1][mask],
                targets[var1][mask]
            )

            loss = loss + INCOHERENT_WEIGHT * criterion(
                corrected[var2][mask],
                targets[var2][mask]
            )

        consistent_mask = ~mask

        if consistent_mask.any():
            loss = loss + COHERENT_WEIGHT * criterion(
                corrected[var1][consistent_mask],
                targets[var1][consistent_mask]
            )

            loss = loss + COHERENT_WEIGHT * criterion(
                corrected[var2][consistent_mask],
                targets[var2][consistent_mask]
            )

    # Zero/nonzero mismatch penalties
    loss = loss + MISMATCH_WEIGHT * (
        zero_nonzero_mismatch(
            corrected["lwc"], corrected["rel"]
        )
        + zero_nonzero_mismatch(
            corrected["iwc"], corrected["rei"]
        )
    )

    # Mean-shift penalties
    mean_shift_penalty = sum(
        torch.mean(corrected[name] - targets[name]) ** 2
        for name in ("lwc", "rel", "iwc", "rei")
    )

    loss = loss + MEAN_SHIFT_WEIGHT * mean_shift_penalty

    return loss

In [ ]:
def apply_cloud_corrections(
    predicted_x,
    y_batch,
    network_liquid,
    network_ice
):
    """
    Correct cloud variables only at inconsistent points.

    Returns
    -------
    predicted_x : torch.Tensor
        Updated atmospheric prediction.

    counts : dict
        Inconsistency counts before and after correction,
        separately for liquid and ice clouds.
    """

    variables = extract_cloud_variables(predicted_x, y_batch)

    mask_liquid = incoherence_mask(
        variables["lwc"],
        variables["rel"],
        EPS_LWC,
        EPS_REL
    )

    mask_ice = incoherence_mask(
        variables["iwc"],
        variables["rei"],
        EPS_IWC,
        EPS_REI
    )

    counts = {
        "liquid_before": mask_liquid.sum().item(),
        "ice_before": mask_ice.sum().item()
    }

    corrected = compute_corrections(
        variables,
        network_liquid,
        network_ice
    )

    # Replace only inconsistent values
    for name in ("lwc", "rel"):
        variables[name][mask_liquid] = corrected[name][mask_liquid]

    for name in ("iwc", "rei"):
        variables[name][mask_ice] = corrected[name][mask_ice]

    mask_liquid_after = incoherence_mask(
        variables["lwc"],
        variables["rel"],
        EPS_LWC,
        EPS_REL
    )

    mask_ice_after = incoherence_mask(
        variables["iwc"],
        variables["rei"],
        EPS_IWC,
        EPS_REI
    )

    counts["liquid_after"] = mask_liquid_after.sum().item()
    counts["ice_after"] = mask_ice_after.sum().item()

    return predicted_x, counts



## 6. Training the Liquid- and Ice-Cloud Correction Networks

Two auxiliary neural networks are trained to correct inconsistencies in retrieved cloud microphysical properties.

Each network has a 5 → 4 → 3 → 2 architecture with Softplus activations, including at the output layer. The liquid-cloud network processes LWC, REL, temperature, water vapour, and pressure, while the ice-cloud network processes IWC, REI, temperature, water vapour, and pressure.

**Training data:** The frozen inverse model is applied to the training observations to retrieve atmospheric states. The auxiliary networks take these predicted states as input and are trained against the corresponding reference atmospheric states.

**Loss function:** The training objective combines:
- Mean Squared Error (MSE) terms for the four cloud variables, with different weights for consistent and inconsistent predictions.
- A Boolean penalty for zero/nonzero mismatches between each pair of cloud variables.
- A mean-shift penalty to reduce systematic differences between corrected and reference cloud properties.

The networks are trained using the Adam optimizer and a learning rate scheduler. Previously saved model parameters are loaded when available.

**Note:** The Boolean zero/nonzero mismatch penalty is non-differentiable and therefore does not contribute to the gradients during training. Additionally, the Softplus output activation produces strictly positive values and does not enforce exact zeros.



In [ ]:
def load_model(model, filename, required=True):
    """
    Load a model state dictionary from disk.

    Parameters
    ----------
    model : torch.nn.Module
        Model to initialize.

    filename : pathlib.Path
        Checkpoint path.

    required : bool
        If True, raise an error when the checkpoint is missing.
    """

    if filename.exists():
        model.load_state_dict(
            torch.load(
                filename,
                map_location=AUX_DEVICE,
                weights_only=True
            )
        )

        print(f"Loaded: {filename}")

    elif required:
        raise FileNotFoundError(
            f"Required checkpoint not found: {filename}"
        )

    else:
        print(f"Initializing new model: {filename}")

    return model.to(AUX_DEVICE)


# ---------------------------------------------------------------
# INITIALIZE CORRECTIVE NETWORKS
# ---------------------------------------------------------------

network_liquid = SimpleNN()
network_ice = SimpleNN()

liquid_checkpoint = MODEL_DIR / "nn1_liq.pth"
ice_checkpoint = MODEL_DIR / "nn1_ice.pth"

# Existing corrective checkpoints are required when
# AUX_EPOCHS = 0, since no corrective training takes place.

require_corrective_checkpoints = (AUX_EPOCHS == 0)

network_liquid = load_model(
    network_liquid,
    liquid_checkpoint,
    required=require_corrective_checkpoints
)

network_ice = load_model(
    network_ice,
    ice_checkpoint,
    required=require_corrective_checkpoints
)

# ---------------------------------------------------------------
# DEFINE OPTIMIZER AND SCHEDULER
# ---------------------------------------------------------------

criterion = nn.MSELoss()

optimizer = optim.Adam(
    list(network_liquid.parameters())
    + list(network_ice.parameters()),
    lr=AUX_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=LR_FACTOR,
    patience=LR_PATIENCE
)

# ---------------------------------------------------------------
# COUNT TRAINING INCONSISTENCIES
# ---------------------------------------------------------------

print("\nComputing original training inconsistencies...")

initial_inconsistencies = 0

with torch.no_grad():

    for x_batch, y_batch in train_loader:

        y_batch = y_batch.to(AUX_DEVICE)

        _, latent_y = model_y(y_batch)

        predicted_x = model_x.decoder(
            mapping_yx(latent_y)
        )

        variables = extract_cloud_variables(
            predicted_x, y_batch
        )

        initial_inconsistencies += count_incoherences(
            variables["lwc"], variables["rel"]
        )

        initial_inconsistencies += count_incoherences(
            variables["iwc"], variables["rei"]
        )

print(
    "Training prediction inconsistencies:",
    initial_inconsistencies
)

# ---------------------------------------------------------------
# OPTIONAL CORRECTIVE-NETWORK TRAINING
# ---------------------------------------------------------------

start_training_time = time.time()

if AUX_EPOCHS == 0:
    print("\nNo corrective training requested.")
    print("Using previously trained corrective networks.")

else:

    print("\nStarting corrective-network training...")

    network_liquid.train()
    network_ice.train()

    for epoch in range(AUX_EPOCHS):

        total_loss = 0.0
        num_incoherences_corr = 0

        for x_batch, y_batch in train_loader:

            x_batch = x_batch.to(AUX_DEVICE)
            y_batch = y_batch.to(AUX_DEVICE)

            optimizer.zero_grad()

            # Frozen inverse-model prediction
            with torch.no_grad():

                _, latent_y = model_y(y_batch)

                predicted_x = model_x.decoder(
                    mapping_yx(latent_y)
                )

            # Predicted and reference cloud variables
            predicted = extract_cloud_variables(
                predicted_x, y_batch
            )

            targets = extract_cloud_variables(
                x_batch, y_batch
            )

            # Inconsistency masks
            mask_liquid = incoherence_mask(
                predicted["lwc"],
                predicted["rel"]
            )

            mask_ice = incoherence_mask(
                predicted["iwc"],
                predicted["rei"]
            )

            # Corrective-network predictions
            corrected = compute_corrections(
                predicted,
                network_liquid,
                network_ice
            )

            if PRINT_BATCH_SHAPES:
                print(
                    "Cloud variable shape:",
                    predicted["lwc"].shape
                )

            # Weighted correction loss
            loss = compute_corrective_loss(
                corrected,
                targets,
                mask_liquid,
                mask_ice,
                criterion
            )

            if loss.item() != 0.0:

                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            # Count mismatches in corrective-network outputs
            num_incoherences_corr += count_incoherences(
                corrected["lwc"], corrected["rel"]
            )

            num_incoherences_corr += count_incoherences(
                corrected["iwc"], corrected["rei"]
            )

        avg_loss = total_loss / len(train_loader)

        scheduler.step(avg_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch [{epoch + 1}/{AUX_EPOCHS}] "
            f"Loss: {avg_loss:.6f} | "
            f"Incoherences: {num_incoherences_corr} | "
            f"LR: {current_lr:.2e}"
        )

    # Save trained corrective networks
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    torch.save(
        network_liquid.state_dict(),
        liquid_checkpoint
    )

    torch.save(
        network_ice.state_dict(),
        ice_checkpoint
    )

    print("Corrective models saved.")

training_time = time.time() - start_training_time

print(f"Training stage duration: {training_time:.2f} s")


## 7. Inference with Cloud Corrections

The auxiliary networks are evaluated at every atmospheric level. However, their outputs replace the cloud microphysical variables predicted by the inverse model **only at levels where inconsistencies are detected in the original predictions**.


In [ ]:
def evaluate_dataset(
    data_loader,
    model_x,
    model_y,
    mapping_xy,
    mapping_yx,
    network_liquid,
    network_ice,
    output_file
):
    """
    Evaluate the frozen inverse model and apply cloud corrections.

    The saved results contain:
        decodedX   : reconstructed atmospheric states
        decodedY   : reconstructed observations
        predictedX : corrected inverse predictions
        predictedY : forward predictions
        originalX  : reference atmospheric states
        originalY  : reference observations

    Corrections affect only predictedX.

    All arrays are saved in sample-by-variable orientation,
    as specified in the input data interface.
    """

    results_batches = {
        "decodedX": [],
        "decodedY": [],
        "predictedX_before": [],  # Inverse predictions before cloud correction
        "predictedX": [],         # Inverse predictions after cloud correction
        "predictedY": [],
        "originalX": [],
        "originalY": []
    }


    counts_total = {
        "liquid_before": 0,
        "liquid_after": 0,
        "ice_before": 0,
        "ice_after": 0
    }

    start_time = time.time()

    with torch.no_grad():

        for x_batch, y_batch in data_loader:

            x_batch = x_batch.to(AUX_DEVICE)
            y_batch = y_batch.to(AUX_DEVICE)

            # Autoencoder reconstructions
            decoded_x, encoded_x = model_x(x_batch)
            decoded_y, encoded_y = model_y(y_batch)

            # Forward and inverse predictions
            predicted_y = model_y.decoder(
                mapping_xy(encoded_x)
            )

            predicted_x = model_x.decoder(
                mapping_yx(encoded_y)
            )

            # Preserve the original inverse predictions before cloud correction
            predicted_x_before = predicted_x.clone()

            # Cloud microphysical correction
            predicted_x, counts = apply_cloud_corrections(
                predicted_x,
                y_batch,
                network_liquid,
                network_ice
            )

            for key in counts_total:
                counts_total[key] += counts[key]

            # Store results on CPU
            batch_results = {
                "decodedX": decoded_x,
                "decodedY": decoded_y,
                "predictedX_before": predicted_x_before,
                "predictedX": predicted_x,
                "predictedY": predicted_y,
                "originalX": x_batch,
                "originalY": y_batch
            }

            for key, value in batch_results.items():
                results_batches[key].append(
                    value.cpu().numpy()
                )

    elapsed_time = time.time() - start_time

    results = {
        key: np.concatenate(values, axis=0)
        for key, values in results_batches.items()
    }

    scipy.io.savemat(output_file, results)

    avg_time_per_batch = elapsed_time / len(data_loader)

    print(f"\nResults saved: {output_file}")
    print(f"Total evaluation time: {elapsed_time:.2f} s")
    print(f"Average time per batch: {avg_time_per_batch:.4f} s")

    print("Liquid inconsistencies:")
    print("  Before:", counts_total["liquid_before"])
    print("  After :", counts_total["liquid_after"])

    print("Ice inconsistencies:")
    print("  Before:", counts_total["ice_before"])
    print("  After :", counts_total["ice_after"])

    return counts_total


### Saving Inference Results

The trained models are evaluated on both the training and test datasets. Results are saved as MATLAB (`.mat`) files in `RESULTS_DIR`.

Each file contains the original atmospheric states and observations (`originalX`, `originalY`), their autoencoder reconstructions (`decodedX`, `decodedY`), the forward predictions (`predictedY`), and the inverse predictions before (`predictedX_before`) and after (`predictedX`) cloud microphysical correction.

The number of cloud inconsistencies before and after correction, together with the evaluation times, is reported in the output.

In [ ]:
network_liquid.eval()
network_ice.eval()
print("Evaluating test dataset...")
test_counts = evaluate_dataset(
    test_loader, model_x, model_y, mapping_xy, mapping_yx,
    network_liquid, network_ice, RESULTS_DIR / "test_set_results_clouds.mat"
)
print("Evaluating training dataset...")
train_counts = evaluate_dataset(
    train_loader, model_x, model_y, mapping_xy, mapping_yx,
    network_liquid, network_ice, RESULTS_DIR / "train_set_results_clouds.mat"
)